# Model-stack benchmark run (Colab T4)

Runs every system and ablation of Phases 4-6 against the model stack: 4-bit
Qwen3-4B-Instruct-2507, bge-m3, bge-reranker-v2-m3 and LLM agents. Results go to Drive under
`model_stack/results/`, **never** `results/`, which holds the offline runs the stability tests
compare against.

1. *Runtime → Change runtime type → T4 GPU.*
2. Run the cells in order. Every run cell passes `--resume`: after a disconnect, re-run the
   setup cell and then the cell that was interrupted. It continues from the last finished
   question and refuses if the configuration changed (DD-064).
3. Paste the smoke-test block (cell 4) back before starting the long runs.

Pinned code version: `v0.7-gpu-run`. No threshold sweep: the LLM controller decides sufficiency itself.

## 1. GPU check

In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
import torch
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU"
print(torch.cuda.get_device_name(0))

## 2. Drive, code at the pinned tag, Tesseract and dependencies

In [ ]:
import os, pathlib, subprocess, sys

REF = "v0.7-gpu-run"   # the pinned tag this run measures (DD-034)
REPO_URL = "https://github.com/manishtiwari2/agentic-rag-pdf"

from google.colab import drive
drive.mount("/content/drive")
DRIVE = pathlib.Path("/content/drive/MyDrive/agentic-pdf-rag")
ROOT = DRIVE / "model_stack"                    # final-table --root; never results/
R = ROOT / "results"
R.mkdir(parents=True, exist_ok=True)
# Model weights on Drive: several GB, downloaded once instead of every session.
os.environ["HF_HOME"] = str(DRIVE / "hf_cache")


def sh(*command):
    print("$", " ".join(map(str, command)), flush=True)
    subprocess.run([str(c) for c in command], check=True)


CODE = pathlib.Path("/content/agentic-pdf-rag")
if not CODE.exists():
    sh("git", "clone", "--depth", "1", "--branch", REF, REPO_URL, CODE)
os.chdir(CODE)
sys.path.insert(0, str(CODE))
sh("apt-get", "install", "-y", "-qq", "tesseract-ocr")
sh(sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt")

## 3. Record which commit produced the run

In [ ]:
commit = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True, check=True).stdout.strip()
tag = subprocess.run(["git", "describe", "--tags", "--exact-match"], capture_output=True, text=True).stdout.strip()
(ROOT / "COMMIT.txt").write_text(f"{commit}\n{tag or 'no tag'}\n")
print("commit", commit, "tag", tag or "NONE")
assert tag == REF, f"HEAD is not the pinned tag {REF}"

## 4. Smoke test: one question, then three benchmark questions

Before hours of runs, check that the stack loads, fits and parses its own output. The block
printed at the end is what to paste back. `llm_parse_failure_rate` above about 0.2 means the LLM
agents are mostly falling back to the rules (DD-055), and the run would not measure what it
claims to.

In [ ]:
import json

sh(sys.executable, "-m", "src.cli", "ask", "--pdf", "benchmark/documents/doc5.pdf",
   "--question", "What problem did FinFET face at 5nm?", "--system", "agentic")
SMOKE = pathlib.Path("/content/smoke")
sh(sys.executable, "-m", "src.cli", "run-benchmark", "--system", "agentic", "--limit", "3",
   "--out", SMOKE)

summary = json.loads((SMOKE / "results.json").read_text())["summary"]
records = json.loads((SMOKE / "per_question.json").read_text())
per_question = summary["latency_median_s"]
# Upper bound: agentic latency for every run (dense and hybrid are faster).
question_runs = 10 * 76 + 3 * 33
print("=" * 30, "PASTE FROM HERE", "=" * 30)
print("answer (q1):           ", records[0]["answer"][:300])
print("citations (q1):        ", records[0]["cited_pages"])
print("peak_vram_gb:          ", summary["peak_vram_gb"])
print("llm_parse_failure_rate:", summary["llm_parse_failure_rate"])
print("seconds per question:  ", per_question)
print(f"estimated full run:     {per_question * question_runs / 3600:.1f} h "
      f"({question_runs} question-runs, agentic latency as an upper bound)")
print("=" * 30, "PASTE TO HERE", "=" * 32)

## 5. The runs

One cell per run, so a disconnect costs one arm at most, and `--resume` recovers even that.

In [ ]:
def run(*flags, out):
    sh(sys.executable, "-m", "src.cli", "run-benchmark", *flags, "--resume", "--out", R / out)

### Baseline A: dense

In [ ]:
run("--system", "dense", out="baseline")

### Baseline B: hybrid

In [ ]:
run("--system", "hybrid", out="hybrid")

### Reranker OFF on Baseline B

In [ ]:
run("--system", "hybrid", "--no-rerank", out="ablations/reranker_off")

### Agentic

In [ ]:
run("--system", "agentic", out="agentic")

### Agentic, planner OFF

In [ ]:
run("--system", "agentic", "--no-planner", out="ablations/planner_off")

### Agentic, hybrid OFF

In [ ]:
run("--system", "agentic", "--no-hybrid", out="ablations/hybrid_off")

### Agentic, reranker OFF

In [ ]:
run("--system", "agentic", "--no-rerank", out="ablations/reranker_off_agentic")

### Agentic, refinement OFF

In [ ]:
run("--system", "agentic", "--no-refinement", out="ablations/refinement_off")

### Agentic, evidence controller OFF

In [ ]:
run("--system", "agentic", "--no-evidence-controller", out="ablations/evidence_controller_off")

### Agentic, verification OFF

In [ ]:
run("--system", "agentic", "--no-verification", out="ablations/verification_off")

### Dev sweep: --max-iterations 1

In [ ]:
run("--system", "agentic", "--max-iterations", "1", "--split", "dev", out="experiments/max_iterations/dev_1")

### Dev sweep: --max-iterations 2

In [ ]:
run("--system", "agentic", "--max-iterations", "2", "--split", "dev", out="experiments/max_iterations/dev_2")

### Dev sweep: --max-iterations 3

In [ ]:
run("--system", "agentic", "--max-iterations", "3", "--split", "dev", out="experiments/max_iterations/dev_3")

## 6. Paired comparisons

The same pairs as Phases 4-6. `compare-runs` refuses a pair that differs in a held-constant
field (EVALUATION_PROTOCOL.md 10); none of these should need `--allow-confounded`.

In [ ]:
def compare(baseline, system, out=None):
    command = [sys.executable, "-m", "src.cli", "compare-runs",
               "--baseline", R / baseline, "--system", R / system]
    if out:
        command += ["--out", R / out]
    sh(*command)

compare("baseline", "hybrid")
compare("baseline", "ablations/reranker_off")
compare("ablations/reranker_off", "hybrid", out="hybrid/comparison_vs_reranker_off.json")
compare("baseline", "agentic")
compare("hybrid", "agentic", out="agentic/comparison_vs_hybrid.json")
compare("agentic", "ablations/planner_off")
compare("agentic", "ablations/hybrid_off")
compare("agentic", "ablations/reranker_off_agentic")
compare("agentic", "ablations/refinement_off")
compare("agentic", "ablations/evidence_controller_off")
compare("agentic", "ablations/verification_off")

## 7. The final table, ablation verdicts and error analysis

In [ ]:
sh(sys.executable, "-m", "src.cli", "final-table", "--root", ROOT)
from IPython.display import Markdown, display
display(Markdown((R / "final" / "REPORT.md").read_text()))

## 8. Bring the results home

Download `MyDrive/agentic-pdf-rag/model_stack/` (or the zip this cell writes) into the
repository root, on a new branch. Follow `NEXT_STEPS.md` from there.

In [ ]:
import shutil
archive = shutil.make_archive(str(DRIVE / "model_stack"), "zip", root_dir=DRIVE, base_dir="model_stack")
print("Wrote", archive)